In [18]:
import dlt
import dlt
from itertools import islice
from dlt.sources.rest_api import rest_api_source
import numpy

def openlibrary_source(query: str = "harry potter"):

    return rest_api_source({
        "client": {
            "base_url": "https://openlibrary.org",
        },
        "resource_defaults": {
            "primary_key": "key",
            "write_disposition": "replace",
        },
        "resources": [
            {
                "name": "books",
                "endpoint": {
                    "path": "search.json",
                    "params": {
                        "q": query,
                        "limit": 100,
                    },
                    "data_selector": "docs",
                    "paginator": {
                        "type": "offset",
                        "limit": 100,
                        "offset_param": "offset",
                        "limit_param": "limit",
                        "total_path": "numFound",
                    },
                },
            },
        ],
    })

pipeline = dlt.pipeline(
    pipeline_name="ol_demo",
    destination="duckdb",
    dataset_name="ol_data",
    progress="log" # logs the pipeline run (Optiona)
)

In [5]:
# Pipeline decomposition: Extract, Normalize, Load

# Extract data from the source
extract_info = pipeline.extract(openlibrary_source())


------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 100.15 MB (42.20%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.59s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 13107200.00/s
Memory usage: 102.02 MB (42.20%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.77s | Rate: 0.00/s
books: 500  | Time: 1.18s | Rate: 424.49/s
Memory usage: 103.90 MB (42.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 4.14s | Rate: 0.00/s
books: 700  | Time: 3.55s | Rate: 197.15/s
Memory usage: 104.52 MB (42.10%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 5.30s | Rate: 0

In [6]:
# print extraction summary
load_id = extract_info.loads_ids[-1]
m = extract_info.metrics[load_id][0]

print("Resources:", list(m["resource_metrics"].keys()))
print("Tables:", list(m["table_metrics"].keys()))
print("Load ID:", load_id)
print()

for resource, rm in m["resource_metrics"].items():
    print(f"Resource: {resource}")
    print(f"rows extracted: {rm.items_count}")
    print()

Resources: ['books']
Tables: ['books']
Load ID: 1776101616.379072

Resource: books
rows extracted: 3757



In [7]:
# Normalize JSON data into a clean relational structure
normalize_info = pipeline.normalize()



------------------- Normalize rest_api in 1776101616.379072 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 113.82 MB (43.90%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1776101616.379072 --------------------
Files: 0/2 (0.0%) | Time: 0.00s | Rate: 0.00/s
Items: 0  | Time: 0.00s | Rate: 0.00/s
Memory usage: 113.82 MB (43.90%) | CPU usage: 0.00%

------------------- Normalize rest_api in 1776101616.379072 --------------------
Files: 13/2 (650.0%) | Time: 0.60s | Rate: 21.80/s
Items: 23662  | Time: 0.60s | Rate: 39704.44/s
Memory usage: 116.57 MB (44.00%) | CPU usage: 0.00%



In [8]:
load_id = normalize_info.loads_ids[-1]
m = normalize_info.metrics[load_id][0]

print("Load ID:", load_id)
print()

print("Tables created/updated:")
for table_name, tm in m["table_metrics"].items():
    # skip dlt internal tables to keep it beginner-friendly
    if table_name.startswith("_dlt"):
        continue
    print(f"  - {table_name}: {tm.items_count} rows")

Load ID: 1776101616.379072

Tables created/updated:
  - books: 3757 rows
  - books__author_key: 4690 rows
  - books__author_name: 4690 rows
  - books__ia: 3726 rows
  - books__ia_collection: 2856 rows
  - books__language: 3773 rows
  - books__series_key: 10 rows
  - books__series_name: 10 rows
  - books__series_position: 10 rows
  - books__id_standard_ebooks: 12 rows
  - books__id_librivox: 67 rows
  - books__id_project_gutenberg: 60 rows


In [10]:
# load data into duckdb
load_info = pipeline.load()

---------------------- Load rest_api in 1776101616.379072 ----------------------
Jobs: 0/13 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 150.83 MB (46.00%) | CPU usage: 0.00%

---------------------- Load rest_api in 1776101616.379072 ----------------------
Jobs: 8/13 (61.5%) | Time: 1.06s | Rate: 7.58/s
Memory usage: 238.83 MB (47.20%) | CPU usage: 0.00%

---------------------- Load rest_api in 1776101616.379072 ----------------------
Jobs: 13/13 (100.0%) | Time: 1.59s | Rate: 8.17/s
Memory usage: 183.41 MB (47.00%) | CPU usage: 0.00%



In [11]:
load_info = pipeline.run(openlibrary_source())

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.00s | Rate: 0.00/s
Memory usage: 170.82 MB (54.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 0.49s | Rate: 0.00/s
books: 100  | Time: 0.00s | Rate: 17476266.67/s
Memory usage: 170.82 MB (54.50%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 1.66s | Rate: 0.00/s
books: 600  | Time: 1.18s | Rate: 510.21/s
Memory usage: 170.82 MB (54.70%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 2.66s | Rate: 0.00/s
books: 1000  | Time: 2.18s | Rate: 459.32/s
Memory usage: 170.82 MB (54.80%) | CPU usage: 0.00%

------------------------------- Extract rest_api -------------------------------
Resources: 0/1 (0.0%) | Time: 3.93s | Rate: 

In [13]:
ds = pipeline.dataset()

In [14]:
ds.tables

['books',
 'books__author_key',
 'books__author_name',
 'books__ia',
 'books__ia_collection',
 'books__language',
 'books__series_key',
 'books__series_name',
 'books__series_position',
 'books__id_standard_ebooks',
 'books__id_librivox',
 'books__id_project_gutenberg',
 '_dlt_version',
 '_dlt_loads',
 '_dlt_pipeline_state']

In [21]:
df = ds.books.df()
df.head(3)

,cover_edition_key,cover_i,ebook_access,edition_count,first_publish_year,has_fulltext,key,lending_edition_s,lending_identifier_s,public_scan_b,title,_dlt_load_id,_dlt_id,subtitle
0,OL61027601M,15155833,borrowable,397,1997,True,/works/OL82563W,OL61057835M,isbn_9757501956008,False,Harry Potter and the Philosopher's Stone,1776103338.1255498,j3Me+AWKWrTDHQ,NaN
1,OL26378158M,15158660,printdisabled,144,2007,True,/works/OL82586W,NaN,NaN,False,Harry Potter and the Deathly Hallows,1776103338.1255498,AAmdpVNJCxnUeQ,NaN
2,OL26234270M,10580435,borrowable,279,1999,True,/works/OL82536W,OL48101764M,bdrc-W8LS66814,False,Harry Potter and the Prisoner of Azkaban,1776103338.1255498,WjfSY05nriO6/Q,NaN
